In [ ]:
import os
import langchain
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter

In [ ]:
NEO4J_URI="REDACTED"
NEO4J_USERNAME="REDACTED"
NEO4J_PASSWORD="REDACTED"

In [ ]:
try:
    graph = Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD
    )
    # Test the connection with a simple query
    graph.query("RETURN 1 as test")
    print("Successfully connected to Neo4j Aura")
except Exception as e:
    print(f"Failed to connect to Neo4j Aura: {e}")

C:\Users\Fahad\AppData\Local\Temp\ipykernel_14476\216573770.py:2: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


Successfully connected to Neo4j Aura


In [ ]:
def load_recipe_files():
    """Load recipes from the 3 uploaded text files"""
    recipes = []

    for i in range(1, 4):
        filename = f"Recipe{i}.txt"
        try:
            with open(filename, 'r', encoding='utf-8') as file:
                lines = file.readlines()

                if len(lines) >= 3:
                    name = lines[0].strip()
                    ingredients_str = lines[1].strip()
                    instructions_str = lines[2].strip()

                    # Parse ingredients (comma-separated)
                    ingredients = [ing.strip() for ing in ingredients_str.split(',')]

                    # Parse instructions (period + comma separated)
                    instructions = [inst.strip() for inst in instructions_str.split('.,')]
                    instructions = [inst.rstrip('.') for inst in instructions]

                    recipes.append({
                        "name": name,
                        "ingredients": ingredients,
                        "steps": instructions
                    })
                    print(f"Loaded: {name}")

        except Exception as e:
            print(f"Error loading {filename}: {e}")

    return recipes

In [ ]:
def demonstrate_text_splitting():
    """Demonstrate different text splitting methods for Part 1"""
    print("PART 1: TEXT SPLITTING DEMONSTRATION")

    # Use content from one of the uploaded files for demonstration
    with open("Recipe1.txt", 'r') as file:
        sample_text = file.read()

    print("\n\nORIGINAL TEXT (first 200 chars):")
    print(sample_text[:200] + "...")

    # 1. Recursive character splitting
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=30
    )
    recursive_splits = recursive_splitter.split_text(sample_text)

    # 2. Token-based splitting
    token_splitter = TokenTextSplitter(
        chunk_size=100,
        chunk_overlap=20
    )
    token_splits = token_splitter.split_text(sample_text)

    print("\n\nRECURSIVE CHARACTER SPLITTING:")
    print(f"Number of chunks: {len(recursive_splits)}")
    for i, chunk in enumerate(recursive_splits[:2]):
        print(f"Chunk {i+1}: {chunk[:100]}...")

    print("\n\nTOKEN-BASED SPLITTING:")
    print(f"Number of chunks: {len(token_splits)}")
    for i, chunk in enumerate(token_splits[:2]):
        print(f"Chunk {i+1}: {chunk[:100]}...")

    return recursive_splits, token_splits

In [ ]:
# Step 7: Neo4j Graph Population
def populate_neo4j_graph(recipes_data):
    """Populate Neo4j with recipe data for Part 3"""
    print("\n\nPART 3: NEO4J GRAPH DATABASE SETUP")

    # Clear existing data
    try:
        graph.query("MATCH (n) DETACH DELETE n")
        print("Cleared existing data")
    except Exception as e:
        print(f"Could not clear data: {e}")

    recipes_added = 0
    for recipe in recipes_data:
        try:
            # Create Recipe node (only name)
            graph.query(
                "MERGE (r:Recipe {name: $name})",
                params={"name": recipe["name"]}
            )

            # Create Ingredient nodes and relationships
            for ingredient in recipe["ingredients"]:
                clean_ingredient = ingredient.strip().lower()
                graph.query(
                    """
                    MATCH (r:Recipe {name: $name})
                    MERGE (i:Ingredient {name: $ingredient})
                    MERGE (r)-[:CONTAINS_INGREDIENT]->(i)
                    """,
                    params={"name": recipe["name"], "ingredient": clean_ingredient}
                )

            # Create Step nodes and relationships
            for order, step in enumerate(recipe["steps"]):
                graph.query(
                    """
                    MATCH (r:Recipe {name: $name})
                    MERGE (s:Step {description: $step, order: $order})
                    MERGE (r)-[:HAS_STEP]->(s)
                    """,
                    params={"name": recipe["name"], "step": step, "order": order}
                )

            print(f"Added recipe: {recipe['name']}")
            recipes_added += 1

        except Exception as e:
            print(f"Error adding recipe {recipe['name']}: {e}")

    # Show schema (FIXED)
    print("GRAPH SCHEMA:")
    try:
        # Use refresh_schema() instead of get_schema()
        graph.refresh_schema()
        print("Schema refreshed successfully")
        # Get the schema as a string
        schema_str = str(graph.schema)
        print(schema_str)
    except Exception as e:
        print(f"Could not get schema: {e}")

    return recipes_added

In [ ]:
# Step 8: Recommendation Engine Setup (FIXED)
def setup_recommendation_engine():
    """Set up the recommendation engine for Part 4"""
    print("\n\nPART 4: GRAPH-BASED RECOMMENDATION ENGINE")

    try:
        llm = Ollama(model="qwen3:4b")
        chain = GraphCypherQAChain.from_llm(
            llm=llm,
            graph=graph,
            verbose=False,
            validate_cypher=True,
            allow_dangerous_requests=True
        )
        print("Recommendation engine setup complete")
        return chain
    except Exception as e:
        print(f"Error setting up recommendation engine: {e}")
        return None

In [ ]:
def run_recommendation_queries(chain):
    """Test the recommendation engine with sample queries"""
    if not chain:
        print("Cannot run queries - chain not available")
        return

    test_queries = [
        "Find recipes that use nuts",
        "What recipes use butter?",
        "Find recipes with chicken",
        "What recipes use cream cheese?",
        "Show me all recipes",
        "Find recipes that use corn"
    ]

    print("RECOMMENDATION ENGINE RESULTS:")
    for i, query in enumerate(test_queries, 1):
        print(f"Query {i}: {query}")
        try:
            result = chain.run(query)
            print(f"Result: {result}")
            print("-" * 50)
        except Exception as e:
            print(f"Error: {e}")
            print("-" * 50)

In [ ]:
def manual_cypher_queries():
    """Show manual Cypher queries for comparison"""
    print("MANUAL CYPHER QUERIES")

    queries = {
        "Recipes with nuts": """
        MATCH (r:Recipe)-[:CONTAINS_INGREDIENT]->(i:Ingredient)
        WHERE i.name CONTAINS 'nut' OR i.name CONTAINS 'pecan'
        RETURN r.name as recipe, COLLECT(i.name) as matching_ingredients
        """,

        "All recipes and their ingredients": """
        MATCH (r:Recipe)-[:CONTAINS_INGREDIENT]->(i:Ingredient)
        RETURN r.name as recipe, COLLECT(i.name) as ingredients
        """,

        "Recipes with cooking steps": """
        MATCH (r:Recipe)-[:HAS_STEP]->(s:Step)
        RETURN r.name as recipe, COUNT(s) as step_count
        """
    }

    for desc, query in queries.items():
        print(f"{desc}:")
        try:
            result = graph.query(query)
            for record in result:
                print(f"  {record}")
        except Exception as e:
            print(f"  Query failed: {e}")

In [ ]:
# Execute all steps in sequence
print("STARTING RECIPE RECOMMENDATION ENGINE SETUP")

# Part 1: Text Splitting
recursive_splits, token_splits = demonstrate_text_splitting()

# Part 2: Load recipes
print("\n\nPART 2: DATA LOADING AND PROCESSING")
recipes = load_recipe_files()
print(f"Loaded {len(recipes)} recipes")

# Part 3: Neo4j Graph Setup
recipes_added = populate_neo4j_graph(recipes)
print(f"Successfully added {recipes_added} recipes to Neo4j Aura")

# Part 4: Recommendation Engine
chain = setup_recommendation_engine()
if chain:
    run_recommendation_queries(chain)

# Manual Cypher examples
manual_cypher_queries()

print("SETUP COMPLETE!")

STARTING RECIPE RECOMMENDATION ENGINE SETUP
PART 1: TEXT SPLITTING DEMONSTRATION
ORIGINAL TEXT (first 200 chars):
No-Bake Nut Cookies
1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
In a h...


RECURSIVE CHARACTER SPLITTING:
Number of chunks: 6
Chunk 1: No-Bake Nut Cookies...
Chunk 2: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans...
TOKEN-BASED SPLITTING:
Number of chunks: 2
Chunk 1: No-Bake Nut Cookies
1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c....
Chunk 2:  milk and butter or margarine., Stir over medium heat until mixture bubbles all over top., Boil and ...
PART 2: DATA LOADING AND PROCESSING
Loaded: No-Bake Nut Cookies
Loaded: Jewell Ball'S Chicken
Loaded: Creamy Corn
Loaded 3 recipes
PART 3: NEO4J GRAPH DATABASE SETUP
Cleared existing data
Added recipe: No-Bake Nut Cookies
Added recipe: Jewell Ball'S Chicken
Added recipe: Creamy Corn
GRAPH SCHEMA:
Schema refreshed successfully
Node properties:
Recipe {name: STRING}
Ingredient {name: STRING}
Step {description: STRING, order: INTEGER}
Relationship properties:

The relationships:
(:Recipe)-[:CONTAINS_INGREDIENT]->(:Ingredient)
(:Reci

C:\Users\Fahad\AppData\Local\Temp\ipykernel_14476\2488103994.py:7: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen3:4b")
C:\Users\Fahad\AppData\Local\Temp\ipykernel_14476\1690460445.py:20: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = chain.run(query)


Result: I don't know the answer.
--------------------------------------------------
Query 2: What recipes use butter?


KeyboardInterrupt: 